# 02 — Tasks et gather

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer des tasks avec `asyncio.create_task()` ;
- exécuter des coroutines en parallèle avec `asyncio.gather()` ;
- utiliser `asyncio.wait()` et `asyncio.wait_for()` ;
- annuler des tasks avec `task.cancel()` ;
- gérer les exceptions dans les tasks.

## Prérequis — ce que vous connaissez déjà

Vous maîtrisez déjà :

- `async def`, `await`, l'event loop ;
- `asyncio.run()` et `asyncio.sleep()` ;
- la distinction coroutine / coroutine function / awaitable.

Notions introduites ici :

- `asyncio.create_task()` et l'objet `Task` ;
- `asyncio.gather()`, `asyncio.wait()`, `asyncio.wait_for()` ;
- annulation et gestion d'erreurs dans les tasks.

## Plan

1. `create_task()` — planifier une coroutine
2. Task vs coroutine nue
3. `asyncio.gather()` — parallélisme simple
4. `asyncio.wait()` — contrôle fin
5. `asyncio.wait_for()` — timeout
6. Annulation de tasks
7. Gestion des exceptions
8. Nommer les tasks
9. Synthèse
10. Exercices

---

## 1. `create_task()` — planifier une coroutine

`asyncio.create_task()` enveloppe une coroutine dans un `Task` et la planifie sur l'event loop. Contrairement à un simple `await`, la task commence son exécution **immédiatement** (au prochain tour de boucle).

In [ ]:
import asyncio

async def tache(nom: str, duree: float) -> str:
    print(f"[{nom}] Début")
    await asyncio.sleep(duree)
    print(f"[{nom}] Fin")
    return f"{nom}: OK"

# Créer deux tasks
task1 = asyncio.create_task(tache("A", 0.5))
task2 = asyncio.create_task(tache("B", 0.3))

# Les tasks s'exécutent en parallèle
r1 = await task1
r2 = await task2
print(f"Résultats : {r1}, {r2}")

In [ ]:
import asyncio

async def demo() -> str:
    await asyncio.sleep(0.1)
    return "fini"

task = asyncio.create_task(demo())
print(f"Type : {type(task)}")
print(f"done() : {task.done()}")
await task
print(f"done() : {task.done()}")
print(f"result() : {task.result()}")

---

## 2. Task vs coroutine nue

| | Coroutine nue (`await coro()`) | Task (`create_task(coro())`) |
|---|---|---|
| Démarrage | Au moment du `await` | Immédiat (prochain tour) |
| Parallélisme | Séquentiel | Concurrent |
| Annulable | Non | Oui (`task.cancel()`) |
| Inspectable | Non | Oui (`.done()`, `.result()`) |

In [ ]:
import asyncio
import time

async def work(n: float) -> float:
    await asyncio.sleep(n)
    return n

# Séquentiel (await direct)
start = time.perf_counter()
await work(0.3)
await work(0.3)
await work(0.3)
print(f"Séquentiel : {time.perf_counter() - start:.2f}s")

# Concurrent (tasks)
start = time.perf_counter()
t1 = asyncio.create_task(work(0.3))
t2 = asyncio.create_task(work(0.3))
t3 = asyncio.create_task(work(0.3))
await t1; await t2; await t3
print(f"Concurrent  : {time.perf_counter() - start:.2f}s")

---

## 3. `asyncio.gather()` — parallélisme simple

`gather()` lance plusieurs awaitables en parallèle et retourne une liste de résultats **dans l'ordre de soumission**.

In [ ]:
import asyncio
import time

async def fetch(url: str) -> str:
    await asyncio.sleep(0.3)
    return f"{url}: 200 OK"

urls = [f"https://api.example.com/{i}" for i in range(5)]

start = time.perf_counter()
resultats = await asyncio.gather(*(fetch(url) for url in urls))
elapsed = time.perf_counter() - start

for r in resultats:
    print(f"  {r}")
print(f"Temps : {elapsed:.2f}s (5 requêtes en ≈0.3s)")

### `return_exceptions=True`

Par défaut, `gather` propage la première exception. Avec `return_exceptions=True`, les exceptions sont retournées dans la liste au lieu d'être levées.

In [ ]:
import asyncio

async def ok(x: int) -> int:
    return x * 2

async def echec(x: int) -> int:
    raise ValueError(f"Erreur pour {x}")

resultats = await asyncio.gather(
    ok(1), echec(2), ok(3),
    return_exceptions=True
)

for r in resultats:
    if isinstance(r, Exception):
        print(f"  ERREUR : {r}")
    else:
        print(f"  OK : {r}")

---

## 4. `asyncio.wait()` — contrôle fin

`wait()` offre plus de contrôle que `gather()`. Il retourne deux ensembles : `done` et `pending`.

| Mode | Comportement |
|---|---|
| `FIRST_COMPLETED` | Retourne dès qu'une task finit |
| `FIRST_EXCEPTION` | Retourne dès qu'une exception survient |
| `ALL_COMPLETED` | Attend toutes les tasks (défaut) |

In [ ]:
import asyncio
import random

async def tache(n: int) -> int:
    duree = random.uniform(0.1, 0.5)
    await asyncio.sleep(duree)
    return n

tasks = [asyncio.create_task(tache(i), name=f"T-{i}") for i in range(5)]

done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
print(f"Terminées : {len(done)}")
print(f"En attente : {len(pending)}")
for t in done:
    print(f"  {t.get_name()} → {t.result()}")

# Attendre les restantes
done2, _ = await asyncio.wait(pending)
for t in done2:
    print(f"  {t.get_name()} → {t.result()}")

---

## 5. `asyncio.wait_for()` — timeout

`wait_for()` attend une coroutine avec un timeout. Si le timeout expire, elle lève `asyncio.TimeoutError` et annule la task.

In [ ]:
import asyncio

async def lente() -> str:
    await asyncio.sleep(5)
    return "fini"

try:
    resultat = await asyncio.wait_for(lente(), timeout=1.0)
except TimeoutError:
    print("Timeout ! La coroutine a pris trop de temps.")

In [ ]:
import asyncio

async def rapide() -> str:
    await asyncio.sleep(0.1)
    return "rapide"

# Pas de timeout
resultat = await asyncio.wait_for(rapide(), timeout=1.0)
print(f"Résultat : {resultat}")

---

## 6. Annulation de tasks

`task.cancel()` demande l'annulation d'une task. La prochaine fois que la task `await` quelque chose, un `asyncio.CancelledError` est levé.

In [ ]:
import asyncio

async def longue_tache() -> None:
    try:
        print("Début de la tâche longue")
        await asyncio.sleep(10)
        print("Fin (ne devrait pas s'afficher)")
    except asyncio.CancelledError:
        print("Tâche annulée ! Nettoyage...")
        raise  # toujours re-raise CancelledError

task = asyncio.create_task(longue_tache())
await asyncio.sleep(0.1)  # laisse la task démarrer
task.cancel()

try:
    await task
except asyncio.CancelledError:
    print(f"Task annulée : {task.cancelled()}")

### Annuler avec un message (Python 3.9+)

In [ ]:
import asyncio

async def worker() -> None:
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError as e:
        print(f"Annulé avec message : {e}")
        raise

task = asyncio.create_task(worker())
await asyncio.sleep(0.1)
task.cancel(msg="timeout dépassé")

try:
    await task
except asyncio.CancelledError:
    pass

---

## 7. Gestion des exceptions

### 7.1. Task qui lève une exception

In [ ]:
import asyncio

async def risquee() -> None:
    await asyncio.sleep(0.1)
    raise ValueError("Quelque chose a mal tourné")

task = asyncio.create_task(risquee())

try:
    await task
except ValueError as e:
    print(f"Exception attrapée : {e}")
    print(f"task.exception() : {task.exception()}")

### 7.2. Tasks "fire-and-forget" : attention aux exceptions silencieuses

Si une task n'est jamais `await`-ée, son exception est perdue (avec un warning). Bonne pratique : toujours `await` ou récupérer l'exception.

In [ ]:
import asyncio

async def fire_and_forget() -> None:
    raise RuntimeError("je suis perdue !")

# Mauvais : exception silencieuse
task = asyncio.create_task(fire_and_forget())
await asyncio.sleep(0.2)
# Python affichera un warning quand le task est garbage collected

# Bon : ajouter un callback pour logger
def handle_exception(t: asyncio.Task) -> None:
    if not t.cancelled() and t.exception():
        print(f"Exception non gérée : {t.exception()}")

task2 = asyncio.create_task(fire_and_forget())
task2.add_done_callback(handle_exception)
await asyncio.sleep(0.2)

---

## 8. Nommer les tasks

Depuis Python 3.8, on peut nommer les tasks pour faciliter le débogage.

In [ ]:
import asyncio

async def worker(n: int) -> int:
    await asyncio.sleep(0.1)
    return n * n

tasks = [
    asyncio.create_task(worker(i), name=f"calcul-{i}")
    for i in range(5)
]

for t in tasks:
    print(f"Task : {t.get_name()}")

resultats = await asyncio.gather(*tasks)
print(f"Résultats : {resultats}")

In [ ]:
import asyncio

# Lister toutes les tasks en cours
async def lister() -> None:
    await asyncio.sleep(0)

tasks = [asyncio.create_task(lister(), name=f"t-{i}") for i in range(3)]

all_tasks = asyncio.all_tasks()
for t in all_tasks:
    print(f"  {t.get_name()} (done={t.done()})")

await asyncio.gather(*tasks)

---

## 9. Synthèse

| Outil | Usage |
|---|---|
| `create_task(coro)` | Planifie une coroutine, retourne un `Task` |
| `gather(*coros)` | Exécute en parallèle, résultats dans l'ordre |
| `wait(tasks)` | Contrôle fin : FIRST_COMPLETED, etc. |
| `wait_for(coro, timeout)` | Timeout sur une coroutine |
| `task.cancel()` | Demande l'annulation |
| `task.result()` | Résultat (ou relève l'exception) |
| `task.done()` | `True` si terminée |
| `return_exceptions=True` | gather retourne les exceptions au lieu de les lever |

**Bonnes pratiques :**

1. Toujours `await` les tasks (ou gérer les exceptions avec un callback).
2. Toujours re-raise `CancelledError` après nettoyage.
3. Nommer les tasks pour le débogage.
4. `gather` pour la simplicité, `wait` pour le contrôle.

---

## 10. Exercices

### Exercice 1 — Téléchargement parallèle *(facile)*

Simuler le téléchargement de 10 fichiers (chacun dure `random.uniform(0.1, 0.5)` secondes). Utiliser `gather()` et afficher chaque résultat. Mesurer le temps total.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Tasks_et_gather", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import random
import time

async def telecharger(nom: str) -> str:
    duree = random.uniform(0.1, 0.5)
    await asyncio.sleep(duree)
    return f"{nom}: {duree:.2f}s"

fichiers = [f"file_{i}.dat" for i in range(10)]

start = time.perf_counter()
resultats = await asyncio.gather(*(telecharger(f) for f in fichiers))
elapsed = time.perf_counter() - start

for r in resultats:
    print(f"  {r}")
print(f"Total : {elapsed:.2f}s")
```

</details>

### Exercice 2 — Premier arrivé *(moyen)*

Simuler 5 requêtes vers des serveurs différents (durées aléatoires). Utiliser `wait(FIRST_COMPLETED)` pour afficher le résultat du premier serveur qui répond, puis annuler les autres.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Tasks_et_gather", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import random

async def requete_serveur(nom: str) -> str:
    duree = random.uniform(0.1, 1.0)
    await asyncio.sleep(duree)
    return f"{nom} a répondu en {duree:.2f}s"

tasks = [
    asyncio.create_task(requete_serveur(f"srv-{i}"), name=f"srv-{i}")
    for i in range(5)
]

done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)

for t in done:
    print(f"Premier : {t.result()}")

# Annuler les restantes
for t in pending:
    t.cancel()

print(f"Annulées : {len(pending)} tasks")
```

</details>

### Exercice 3 — Retry async *(moyen)*

Écrire une coroutine `retry_async(coro_factory, max_retries=3, delay=0.5)` qui :

1. Appelle `coro_factory()` pour obtenir une coroutine.
2. L'`await` dans un `wait_for` avec timeout de 2s.
3. En cas d'exception ou timeout, attend `delay` puis réessaie.
4. Après `max_retries`, propage la dernière exception.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Tasks_et_gather", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import random
from collections.abc import Callable, Coroutine
from typing import Any

async def retry_async(
    coro_factory: Callable[[], Coroutine[Any, Any, Any]],
    max_retries: int = 3,
    delay: float = 0.5,
    timeout: float = 2.0,
) -> Any:
    last_exc: BaseException | None = None
    for attempt in range(1, max_retries + 1):
        try:
            return await asyncio.wait_for(coro_factory(), timeout=timeout)
        except (TimeoutError, Exception) as e:
            last_exc = e
            print(f"  Tentative {attempt}/{max_retries} : {e}")
            if attempt < max_retries:
                await asyncio.sleep(delay)
    raise last_exc

# Test
compteur = 0
async def flaky() -> str:
    global compteur
    compteur += 1
    if compteur < 3:
        raise ConnectionError("Réseau instable")
    return "Succès !"

compteur = 0
r = await retry_async(flaky, max_retries=5, delay=0.1)
print(f"Résultat : {r}")
```

</details>

### Exercice 4 — Sémaphore async *(difficile)*

Simuler 20 requêtes API concurrentes mais limiter à 3 simultanées avec `asyncio.Semaphore(3)`. Mesurer le temps et vérifier qu'il est proche de `ceil(20/3) * delai`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Tasks_et_gather", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import time

sem = asyncio.Semaphore(3)

async def requete(i: int) -> str:
    async with sem:
        await asyncio.sleep(0.3)
        return f"req-{i}: OK"

start = time.perf_counter()
resultats = await asyncio.gather(*(requete(i) for i in range(20)))
elapsed = time.perf_counter() - start

print(f"Requêtes terminées : {len(resultats)}")
print(f"Temps : {elapsed:.2f}s (attendu ≈ {(20 // 3 + 1) * 0.3:.1f}s)")
```

</details>

---

## Ressources

- [docs Python — asyncio Tasks](https://docs.python.org/3/library/asyncio-task.html)
- [RealPython — asyncio gather](https://realpython.com/python-async-features/)
- [PEP 3156 — Asynchronous IO Support](https://peps.python.org/pep-3156/)